# Лабораторная работа 4: Linear Discriminant Analysis в Python

Задача: по химическим признакам вина `C1–C13` определить производителя `V1`, `V2`, `V3`.

В ноутбуке реализованы два варианта:

1. **LINEST-аналог**, как в Excel через линейную модель.
2. **Solver-аналог**, где коэффициенты оптимизируются по отношению `Inter-group variance / Within-group variance`.

Без `sklearn`, чтобы было видно, что происходит под капотом. Да, страдаем честно.


In [ ]:
import pandas as pd
import numpy as np

file_path = "Chapter4-HW(2).xlsx"

raw = pd.read_excel(file_path, sheet_name="LDA")

# Берем только исходные данные: Vendor + C1:C13
data = raw.iloc[:, :14].copy()

# Обучающие строки: известные производители V1, V2, V3
train = data[data["Vendor"].isin(["V1", "V2", "V3"])].copy()

# Строки для классификации: Vendor = ?
unknown = data[data["Vendor"].eq("?")].copy()

features = [f"C{i}" for i in range(1, 14)]

train.head()


In [ ]:
# Кодируем классы как в Excel:
# V1 -> 0
# V2 -> 1
# V3 -> 2

class_to_num = {
    "V1": 0,
    "V2": 1,
    "V3": 2
}

num_to_class = {
    0: "V1",
    1: "V2",
    2: "V3"
}

X = train[features].astype(float).values
y = train["Vendor"].map(class_to_num).astype(float).values

# Добавляем столбец единиц для свободного коэффициента b
X_design = np.column_stack([np.ones(len(X)), X])

print("Размер обучающей выборки:", X.shape)
print("Количество объектов по классам:")
print(train["Vendor"].value_counts().sort_index())


## 1. LINEST-аналог

В Excel использовалась функция `ЛИНЕЙН`.

В Python то же самое можно сделать методом наименьших квадратов:

\[
y = b + w_1x_1 + w_2x_2 + ... + w_{13}x_{13}
\]


In [ ]:
# Аналог Excel LINEST
linest_coefficients = np.linalg.lstsq(X_design, y, rcond=None)[0]

coef_names = ["b"] + [f"w{i}" for i in range(1, 14)]

linest_table = pd.DataFrame({
    "coefficient": coef_names,
    "value": linest_coefficients
})

linest_table


In [ ]:
def calculate_predictions(X_design, coefficients):
    return X_design @ coefficients


def calculate_means_counts_cutoffs(predictions, y):
    result_df = pd.DataFrame({
        "y": y,
        "prediction": predictions
    })

    means = result_df.groupby("y")["prediction"].mean()
    counts = result_df.groupby("y")["prediction"].count()

    cutoff_01 = (means.loc[0] * counts.loc[0] + means.loc[1] * counts.loc[1]) / (counts.loc[0] + counts.loc[1])
    cutoff_12 = (means.loc[1] * counts.loc[1] + means.loc[2] * counts.loc[2]) / (counts.loc[1] + counts.loc[2])

    return means, counts, [cutoff_01, cutoff_12]


def classify_by_cutoffs(predictions, cutoffs):
    classes = []

    for value in predictions:
        if value < cutoffs[0]:
            classes.append("V1")
        elif value < cutoffs[1]:
            classes.append("V2")
        else:
            classes.append("V3")

    return np.array(classes)


def count_errors(real_classes, predicted_classes):
    return (real_classes.values != predicted_classes).sum()


def lda_metrics(coefficients, X_design, y):
    predictions = calculate_predictions(X_design, coefficients)
    means, counts, cutoffs = calculate_means_counts_cutoffs(predictions, y)

    inter_group = ((means - means.mean()) ** 2).sum()

    within_group = 0
    for class_number in [0, 1, 2]:
        class_predictions = predictions[y == class_number]
        within_group += ((class_predictions - means.loc[class_number]) ** 2).sum()

    ratio = inter_group / within_group

    return {
        "predictions": predictions,
        "means": means,
        "counts": counts,
        "cutoffs": cutoffs,
        "inter_group": inter_group,
        "within_group": within_group,
        "ratio": ratio
    }


In [ ]:
linest_metrics = lda_metrics(linest_coefficients, X_design, y)

linest_train_predictions = linest_metrics["predictions"]
linest_cutoffs = linest_metrics["cutoffs"]
linest_predicted_classes = classify_by_cutoffs(linest_train_predictions, linest_cutoffs)

linest_difference = count_errors(train["Vendor"], linest_predicted_classes)

print("LINEST means:")
print(linest_metrics["means"])

print("\nLINEST sample numbers:")
print(linest_metrics["counts"])

print("\nLINEST cutoffs:")
print(linest_cutoffs)

print("\nLINEST Difference =", linest_difference)
print("Inter-group variance =", linest_metrics["inter_group"])
print("Within-group variance =", linest_metrics["within_group"])
print("Inter/within ratio =", linest_metrics["ratio"])


In [ ]:
# Классификация неизвестных строк 182-184 через LINEST

X_unknown = unknown[features].astype(float).values
X_unknown_design = np.column_stack([np.ones(len(X_unknown)), X_unknown])

unknown_predictions_linest = calculate_predictions(X_unknown_design, linest_coefficients)
unknown_classes_linest = classify_by_cutoffs(unknown_predictions_linest, linest_cutoffs)

unknown_result_linest = unknown[["Vendor"] + features].copy()
unknown_result_linest["Numerical prediction"] = unknown_predictions_linest
unknown_result_linest["Predicted vendor"] = unknown_classes_linest

unknown_result_linest


## 2. Solver-аналог

Теперь оптимизируем коэффициенты так, как Solver в Excel: максимизируем отношение

\[
\frac{S_G}{S_W}
\]

где:

- \(S_G\) — межгрупповая дисперсия;
- \(S_W\) — внутригрупповая дисперсия.

В Excel это делалось через Solver. В Python используем `scipy.optimize`.


In [ ]:
from scipy.optimize import minimize

def objective(coefficients):
    metrics = lda_metrics(coefficients, X_design, y)
    return -metrics["ratio"]  # maximize ratio -> minimize negative ratio


# Стартуем с коэффициентов LINEST, как в Excel
start_coefficients = linest_coefficients.copy()

solver_result = minimize(
    objective,
    start_coefficients,
    method="BFGS",
    options={
        "maxiter": 10000,
        "gtol": 1e-9
    }
)

solver_coefficients = solver_result.x

solver_table = pd.DataFrame({
    "coefficient": coef_names,
    "LINEST": linest_coefficients,
    "Solver-like": solver_coefficients
})

solver_table


In [ ]:
solver_metrics = lda_metrics(solver_coefficients, X_design, y)

solver_train_predictions = solver_metrics["predictions"]
solver_cutoffs = solver_metrics["cutoffs"]
solver_predicted_classes = classify_by_cutoffs(solver_train_predictions, solver_cutoffs)

solver_difference = count_errors(train["Vendor"], solver_predicted_classes)

print("Solver-like means:")
print(solver_metrics["means"])

print("\nSolver-like sample numbers:")
print(solver_metrics["counts"])

print("\nSolver-like cutoffs:")
print(solver_cutoffs)

print("\nSolver-like Difference =", solver_difference)
print("Inter-group variance =", solver_metrics["inter_group"])
print("Within-group variance =", solver_metrics["within_group"])
print("Inter/within ratio =", solver_metrics["ratio"])


In [ ]:
# Классификация неизвестных строк 182-184 через Solver-like коэффициенты

unknown_predictions_solver = calculate_predictions(X_unknown_design, solver_coefficients)
unknown_classes_solver = classify_by_cutoffs(unknown_predictions_solver, solver_cutoffs)

unknown_result_solver = unknown[["Vendor"] + features].copy()
unknown_result_solver["Numerical prediction"] = unknown_predictions_solver
unknown_result_solver["Predicted vendor"] = unknown_classes_solver

unknown_result_solver


## 3. Сравнение итогов

Главный результат задания — классификация неизвестных вин в строках 182–184.


In [ ]:
comparison = pd.DataFrame({
    "Row": unknown.index + 2,  # Excel rows
    "LINEST prediction": unknown_predictions_linest,
    "LINEST vendor": unknown_classes_linest,
    "Solver prediction": unknown_predictions_solver,
    "Solver vendor": unknown_classes_solver
})

comparison


In [ ]:
# Сохраняем результат в Excel

output_file = "Chapter4_HW_LDA_Python_Result.xlsx"

with pd.ExcelWriter(output_file) as writer:
    linest_table.to_excel(writer, sheet_name="LINEST coefficients", index=False)
    solver_table.to_excel(writer, sheet_name="Solver coefficients", index=False)
    comparison.to_excel(writer, sheet_name="Unknown classification", index=False)

print(f"Результат сохранен в файл: {output_file}")
